# Paper 4 — 06 · Aggregate + figures + LaTeX

Collect every per-anchor JSON from `results/`, build the headline table, the per-layer transfer-drop figure, the patching restoration curve, the SAE firing figure, the Paper-3 band-map figure, and the mechanistic-vs-behavioral correlation scatter. Emit LaTeX fragments for the manuscript. No GPU.

**Output:** `figures/*.pdf`, `manuscript/tables/*.tex`.

In [ ]:
%%capture
# Colab already ships consistent torch / matplotlib / pandas / scipy. We add
# only what's genuinely missing or needs a newer pin. Deliberately we do NOT
# `-U matplotlib` (upgrading it mid-session breaks the PDF backend: 'cannot
# import name FontPath'), and we do NOT install transformer-lens / nnsight /
# seaborn (unused). sae-lens is installed only in nb04 (the one place it's used).
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'datasets>=3.0' \
    'scikit-learn>=1.4' \
    python-dotenv requests huggingface_hub ipywidgets pyyaml -q


In [ ]:
import os, json, gc, sys, hashlib, subprocess
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Secrets (Colab) -> env, so the Paper 2 judge + gated HF models work
#     end-to-end with no manual steps. Set these in Colab -> Secrets first. ---
try:
    from google.colab import userdata
    for _k in ("OPENROUTER_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_k)
            if _v:
                os.environ[_k] = _v
        except Exception:
            print(f"[secrets] {_k} not set in Colab Secrets — add it if a cell needs it.")
except Exception:
    pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)

# --- Artifact root (persistent, on Drive) ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper4-interpretability")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")
PAPER3_ROOT = Path("/content/drive/MyDrive/PhD/paper3-alignment")

# --- Code root: use the repo synced on Drive if present, else clone the public
#     repo to /content. Self-provisioning AND self-updating: if the /content
#     clone already exists we `git pull` it, so you always get the latest code. ---
REPO_URL = "https://github.com/robery567/rosafety-circuits.git"
if (DRIVE_ROOT / "src" / "paths.py").exists():
    CODE_ROOT = DRIVE_ROOT
else:
    CODE_ROOT = Path("/content/rosafety-circuits")
    if (CODE_ROOT / ".git").exists():
        print("Updating Paper 4 code (git pull):", CODE_ROOT)
        subprocess.run(["git", "-C", str(CODE_ROOT), "pull", "-q", "--ff-only"], check=False)
    else:
        print("Cloning Paper 4 code:", REPO_URL)
        subprocess.run(["git", "clone", "-q", REPO_URL, str(CODE_ROOT)], check=True)
print("CODE_ROOT :", CODE_ROOT)
print("DRIVE_ROOT:", DRIVE_ROOT)

# --- data dirs (Drive, persistent across sessions) ---
DATA_DIR     = DRIVE_ROOT / "data"
CONTRAST_DIR = DATA_DIR / "contrastive"
ACT_DIR      = DATA_DIR / "activations"
PROBE_DIR    = DATA_DIR / "probes"
SPLITS_DIR   = DATA_DIR / "splits"
RESULTS_DIR  = DRIVE_ROOT / "results"
FIG_DIR      = DRIVE_ROOT / "figures"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [CONTRAST_DIR, ACT_DIR, PROBE_DIR, SPLITS_DIR, RESULTS_DIR, FIG_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
CONFIG_DIR = CODE_ROOT / "configs"   # configs live in the repo, not in data/

# --- Reuse Paper 2 judge harness; Paper 4 src/ from CODE_ROOT ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))      # judges.py, llm_judge.py
sys.path.insert(0, str(CODE_ROOT / "src"))         # paths, capture, probes, patching, sae_utils, contrastive, behavioral

# Drop any cached Paper 4 modules so a fresh import picks up a just-pulled
# version without needing a kernel restart.
for _m in ("paths", "capture", "probes", "patching", "sae_utils", "contrastive", "behavioral"):
    sys.modules.pop(_m, None)
from paths import savefig   # robust multi-format figure saver (PDF->cairo->SVG->PNG)

# --- A100 sanity ---
assert torch.cuda.is_available(), "Need a GPU runtime (A100 high-RAM)."
torch.backends.cuda.matmul.allow_tf32 = True
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)


## 1. Load all per-anchor results

In [ ]:
import glob
shorts = sorted({Path(p).parent.name for p in glob.glob(str(RESULTS_DIR / '*' / '*.json'))})
def load(short, name):
    f = RESULTS_DIR / short / f'{name}.json'
    return json.loads(f.read_text()) if f.exists() else None
R = {s: {n: load(s, n) for n in ['linear_probes','activation_patching','sae_features','paper3_crossref']} for s in shorts}
print('anchors with results:', shorts)

## 2. Headline table (H1a/H1b/H1c/H1d per anchor)

In [ ]:
import numpy as np, pandas as pd
rows = []
for s in shorts:
    lp, ap, x3 = R[s]['linear_probes'], R[s]['activation_patching'], R[s]['paper3_crossref']
    row = {'anchor': s}
    if lp:
        b = lp['bands']; pl = {p['layer']: p for p in lp['per_layer']}
        row['det_drop@detection'] = float(np.mean([pl[L]['det_drop'] for L in b['detection']]))
        row['exe_drop@execution'] = float(np.mean([pl[L]['exe_drop'] for L in b['execution']]))
    if ap:
        row['patch_peak_band'] = ap['peak_band']; row['patch_peak_restoration'] = {p['layer']: p for p in ap['per_layer']}[ap['peak_layer']]['restoration']
    if x3:
        row['H1d_blocks_execution'] = x3['h1d_blocks_are_execution']
    rows.append(row)
df = pd.DataFrame(rows).set_index('anchor')
df

## 3. Cross-anchor figure: detection vs execution transfer drop in-band

In [ ]:
import matplotlib.pyplot as plt
if 'det_drop@detection' not in df.columns:
    print('No linear_probes results yet (run nb02) — skipping transfer-drop figure.')
else:
    fig, ax = plt.subplots(figsize=(6,4))
    x = np.arange(len(df)); w = 0.38
    ax.bar(x-w/2, df['det_drop@detection'], w, label='detection drop @ detection band')
    ax.bar(x+w/2, df['exe_drop@execution'], w, label='execution drop @ execution band')
    ax.set_xticks(x); ax.set_xticklabels(df.index, rotation=15); ax.set_ylabel('EN->RO accuracy drop')
    ax.legend(fontsize=8); ax.set_title('H1a/H1b: detection drop >> execution drop')
    fig.tight_layout(); savefig(fig, FIG_DIR / 'summary_transfer_drop.pdf'); plt.show()

## 4. Mechanistic-vs-behavioral correlation (cross-anchor; small-n, report ρ)

In [ ]:
from scipy.stats import spearmanr
# detection-band drop vs the Paper 2 behavioral RO gap per anchor (read from models.yaml baselines).
import yaml
mdl = yaml.safe_load((CONFIG_DIR / 'models.yaml').read_text())
base = {m['short']: m.get('paper2_baseline', {}) for m in mdl.get('anchors', [])}
xs, ys, labs = [], [], []
for s in shorts:
    if s in base and base[s] and not np.isnan(df.loc[s].get('det_drop@detection', np.nan)):
        xs.append(1 - base[s]['tox']); ys.append(df.loc[s]['det_drop@detection']); labs.append(s)
if len(xs) >= 3:
    rho, p = spearmanr(xs, ys); print(f'Spearman rho={rho:.2f} p={p:.3f} (n={len(xs)})')
else:
    print(f'n={len(xs)} anchors with both signals — need >=3 for a correlation.')

## 5. Emit LaTeX headline table (manuscript/tables/ — gitignored)

In [ ]:
tdir = DRIVE_ROOT / 'manuscript' / 'tables'; tdir.mkdir(parents=True, exist_ok=True)
(tdir / 'headline.tex').write_text(df.round(3).to_latex())
print('wrote', tdir / 'headline.tex')
print(df.round(3).to_string())